In [ ]:
!pip install anthropic requests python-dotenv -q

import os
import anthropic
import requests
import json
import time
from dotenv import load_dotenv

load_dotenv()

GITHUB_TOKEN = os.environ["GITHUB_TOKEN"]
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
REPO = "PaperMC/Paper"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def github_api(endpoint, params=None, max_retries=3):
    url = f"https://api.github.com/repos/{REPO}/{endpoint}"
    headers = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github.v3+json"
    }
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, params=params)
            if response.status_code == 403:
                print("  ⏳ Rate limit hit. Sleeping 60s...")
                time.sleep(60)
                continue
            if response.status_code >= 500:
                wait_time = 2 ** attempt
                print(f"  ⚠️ GitHub Server Error ({response.status_code}). Retrying in {wait_time}s... ({attempt+1}/{max_retries})")
                time.sleep(wait_time)
                continue
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            wait_time = 2 ** attempt
            print(f"  🚨 Network Exception: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)
    print(f"  ❌ Failed after {max_retries} attempts. Skipping.")
    return {}

print("✅ 초기화 완료!")


In [ ]:
def pass_1_mechanical_screening():
    print("Initiating Pass 1: Mechanical Screening...")
    start = time.time()

    issues = github_api("issues", params={"state": "closed", "sort": "comments", "per_page": 100})
    qualified_candidates = []

    for issue in issues:
        issue_num = issue['number']

        # 1. more than 8 comments
        if issue.get("comments", 0) < 8:
            continue

        # 2. check for linked pr and if it's merged
        pr_data = issue.get("pull_request")
        if not pr_data:
            print(f"  [Skip #{issue_num}] No linked PR")
            continue

        pr_num = int(pr_data["url"].split("/")[-1])
        pr_details = github_api(f"pulls/{pr_num}")

        if not pr_details.get("merged"):
            print(f"  [Skip #{issue_num}] PR is not merged")
            continue

        # 3. changed files count 5 or less
        if pr_details.get("changed_files", 0) > 5:
            print(f"  [Skip #{issue_num}] Too many file changes ({pr_details.get('changed_files')}개)")
            continue

        # 4. participant count 3 or more
        comments = github_api(f"issues/{issue_num}/comments", params={"per_page": 100})
        authors = {c['user']['login'] for c in comments if c.get('user')}
        if issue.get('user'):
            authors.add(issue['user']['login'])

        if len(authors) < 3:
            print(f"  [Skip #{issue_num}] Not enough discussion participants")
            continue

        # passes all mechanical filters, gather metadata for next pass
        merged_at = pr_details.get("merged_at", "")
        reviewers = list({
            r['user']['login']
            for r in github_api(f"pulls/{pr_num}/reviews")
            if r.get('user')
        })

        print(f"  ✅ [Pass #{issue_num}] {issue['title'][:50]}")
        qualified_candidates.append({
            "issue": issue,
            "pr_num": pr_num,
            "comments": comments,
            "pr_meta": {
                "merged_at": merged_at,
                "reviewers": reviewers,
                "changed_files": pr_details.get("changed_files", 0)
            }
        })
        time.sleep(0.5)

    elapsed = round(time.time() - start, 1)
    print(f"\nPass 1 Complete. {len(qualified_candidates)} candidates survived. ({elapsed}s)")
    return qualified_candidates


def pass_1_5_semantic_gate(qualified_candidates):
    print("\nInitiating Pass 1.5: Semantic Pre-Screening Gate...")
    start = time.time()
    elite_candidates = []

    for candidate in qualified_candidates:
        issue = candidate['issue']
        comments_snippet = "\n".join([
            f"{c.get('user', {}).get('login', 'unknown')} ({c.get('author_association', 'NONE')}): {c.get('body', '')[:200]}"
            for c in candidate['comments'][:5]
        ])
        compact_context = f"Title: {issue['title']}\nBody: {str(issue.get('body', ''))[:400]}\nComments:\n{comments_snippet}"
        prompt = (
            "Does this open-source discussion contain technical friction, architectural trade-offs, "
            "or design conflicts? Respond strictly with Yes or No.\n\n"
            f"Context:\n{compact_context}"
        )

        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=10,
                temperature=0,
                messages=[{"role": "user", "content": prompt}]
            )
            decision = response.content[0].text.strip()
            if "Yes" in decision:
                print(f"  💎 #{issue['number']}: {issue['title'][:50]}")
                elite_candidates.append(candidate)
            else:
                print(f"  ⏭️  #{issue['number']} skipped by semantic gate.")
        except Exception as e:
            print(f"  🚨 API Error on #{issue['number']}: {e}")
        time.sleep(0.5)

    elapsed = round(time.time() - start, 1)
    print(f"\nPass 1.5 Complete. {len(elite_candidates)} elite candidates secured. ({elapsed}s)")
    return elite_candidates


candidates_p1 = pass_1_mechanical_screening()
elite_p2 = pass_1_5_semantic_gate(candidates_p1)

import pickle
with open("elite_candidates.pkl", "wb") as f:
    pickle.dump(elite_p2, f)
print("✅ elite_candidates.pkl saved!")

Initiating Pass 1: Mechanical Screening...
  [Skip #2572] PR이 Merge 되지 않음
  [Skip #8108] 파일 변경 너무 많음 (671개)
  [Skip #8235] 파일 변경 너무 많음 (30개)
  [Skip #8058] PR이 Merge 되지 않음
  [Skip #1223] 연결된 PR 없음
  [Skip #2308] 파일 변경 너무 많음 (35개)
  [Skip #9614] PR이 Merge 되지 않음
  [Skip #4902] PR이 Merge 되지 않음
  [Skip #349] PR이 Merge 되지 않음
  [Skip #33] PR이 Merge 되지 않음
  [Skip #4702] PR이 Merge 되지 않음
  ✅ [Pass #1397] Async Chunk Loading and Generation
  [Skip #8711] PR이 Merge 되지 않음
  ✅ [Pass #12589] Improve outdated version check
  ✅ [Pass #11810] Extend HumanEntity#dropItem API
  [Skip #8177] 파일 변경 너무 많음 (942개)
  ✅ [Pass #10036] Scoreboard objective number format api 
  ✅ [Pass #2619] Add Mob Goal API
  [Skip #6308] 파일 변경 너무 많음 (19개)
  [Skip #4101] PR이 Merge 되지 않음
  ✅ [Pass #6562] Teleportation API
  [Skip #4123] 파일 변경 너무 많음 (645개)
  [Skip #138] 연결된 PR 없음
  [Skip #10253] PR이 Merge 되지 않음
  [Skip #3106] PR이 Merge 되지 않음
  [Skip #895] 연결된 PR 없음
  ✅ [Pass #6318] Hide unnecessary itemmeta from clients
  [Skip #5

In [6]:
def pass_2_deep_enrichment(elite_candidates):
    print("\nInitiating Pass 2: Deep Data Enrichment...")
    start = time.time()
    master_bundles = []

    for candidate in elite_candidates:
        issue = candidate["issue"]
        pr_num = candidate["pr_num"]
        issue_num = issue["number"]

        # 1. Discussion Timeline (댓글 전체, 코드 제외)
        discussion_timeline = []
        for comment in candidate["comments"]:
            discussion_timeline.append({
                "author": comment["user"]["login"],
                "role": comment.get("author_association", "NONE"),
                "body": comment["body"],
                "timestamp": comment["created_at"]
            })

        # 2. Commit History (메시지만, 코드 없음)
        commits_data = github_api(f"pulls/{pr_num}/commits")
        commit_history = []
        for commit in commits_data:
            commit_history.append({
                "sha": commit["sha"][:7],
                "message": commit["commit"]["message"],
                "author": commit["commit"]["author"]["name"],
                "timestamp": commit["commit"]["author"]["date"]
            })

        # 3. Master Bundle 조립 (코드 없는 깔끔한 구조)
        bundle = {
            "story_id": f"{REPO}#{issue_num}",
            "metadata": {
                "title": issue["title"],
                "labels": [l["name"] for l in issue.get("labels", [])],
                "target_course": "Java OOP / Systems Engineering"
            },
            "pr_metadata": candidate["pr_meta"],
            "issue_payload": {
                "body": issue.get("body", "")
            },
            "discussion_timeline": discussion_timeline,
            "commit_history": commit_history
        }

        master_bundles.append(bundle)
        print(f"  ✅ #{issue_num}: {issue['title'][:50]} | 댓글 {len(discussion_timeline)}개 | 커밋 {len(commit_history)}개")
        time.sleep(1)

    with open("raw_story_bundles.json", "w", encoding="utf-8") as f:
        json.dump(master_bundles, f, indent=2, ensure_ascii=False)

    elapsed = round(time.time() - start, 1)
    print(f"\nPipeline Complete. Exported {len(master_bundles)} Master Bundles. ({elapsed}s)")
    return master_bundles


final_bundles = pass_2_deep_enrichment(elite_p2[:5])


Initiating Pass 2: Deep Data Enrichment...
  ✅ #1397: Async Chunk Loading and Generation | 댓글 79개 | 커밋 30개
  ✅ #12589: Improve outdated version check | 댓글 23개 | 커밋 30개
  ✅ #11810: Extend HumanEntity#dropItem API | 댓글 47개 | 커밋 23개
  ✅ #10036: Scoreboard objective number format api  | 댓글 18개 | 커밋 25개
  ✅ #2619: Add Mob Goal API | 댓글 40개 | 커밋 1개
  ✅ #6562: Teleportation API | 댓글 39개 | 커밋 17개
  ✅ #6318: Hide unnecessary itemmeta from clients | 댓글 37개 | 커밋 2개
  ✅ #9209: Add player whitelist events | 댓글 21개 | 커밋 11개
  ✅ #4722: Player Entity Tracking Events | 댓글 32개 | 커밋 1개
  ✅ #3544: Fix piston physics inconsistency (fix tnt dupers) | 댓글 53개 | 커밋 2개
  ✅ #6356: Add CompostItemEvent and EntityCompostItemEvent | 댓글 12개 | 커밋 1개
  ✅ #6278: Add System.out.println catcher | 댓글 31개 | 커밋 1개
  ✅ #728: Improve console implementation | 댓글 10개 | 커밋 1개
  ✅ #13455: Fix memory leak on constantly damage | 댓글 35개 | 커밋 7개
  ✅ #12273: Add configuration interface to expose certain conf | 댓글 14개 | 커밋 15개
  ✅ #32

In [7]:
import json

with open("raw_story_bundles.json", "r") as f:
    bundles = json.load(f)

# #1397 번들 꺼내기
bundle = bundles[0]

# LLM에게 먹일 컨텍스트 조립
issue_body = bundle["issue_payload"]["body"]
timeline = "\n\n".join([
    f"[{c['timestamp'][:10]}] {c['author']} ({c['role']}):\n{c['body'][:500]}"
    for c in bundle["discussion_timeline"]
])
commits = "\n".join([
    f"- [{c['timestamp'][:10]}] {c['author']}: {c['message'][:100]}"
    for c in bundle["commit_history"]
])

prompt = f"""You are an expert software engineering educator analyzing a real open-source GitHub discussion.

Below is the complete engineering narrative of a real Pull Request from the PaperMC/Paper repository.

=== ISSUE TITLE ===
{bundle["metadata"]["title"]}

=== PROBLEM DEFINITION ===
{issue_body[:1000]}

=== DEVELOPER DISCUSSION TIMELINE ===
{timeline}

=== COMMIT HISTORY ===
{commits}

---

Based on the above, please analyze and extract the following:

1. **The Core Problem**: What was the fundamental technical challenge being solved?
2. **The Key Debate**: What were the main points of technical friction or disagreement between developers?
3. **The Architectural Decision**: What was the final design decision and why was it chosen over alternatives?
4. **The Engineering Lessons**: What are 2-3 key software engineering lessons a student could learn from this discussion?
"""

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=[{"role": "user", "content": prompt}]
)

print(response.content[0].text)

# Analysis of PaperMC Async Chunk Loading PR

## 1. The Core Problem

The fundamental challenge was **moving Minecraft's chunk loading and generation off the main server thread** without breaking the game's core invariants.

Minecraft's server architecture is inherently single-threaded — virtually all game state mutations (entities, blocks, lighting, structures) assume they execute sequentially on one thread. The problem manifested in several concrete ways:

- **I/O blocking the game loop**: Chunk loading from disk and chunk generation (which involves heavy CPU work like terrain, structure, and feature generation) happened synchronously on the main thread, causing tick lag spikes whenever players explored new areas or teleported
- **Mojang's threading assumptions**: The codebase actively *crashed* when it detected writes from multiple threads (`"Writing from multiple threads"` error), and data structures like `DataFixer`s were not thread-safe
- **Entity lifecycle coupling**: Entities s